# Level 2, Notebook 03: Access Control and Audit

**Problems in this notebook**

- Problem 12: Filter RAG Chunks by User Permissions
- Problem 17: Audit Log Generation

**How to use this notebook**

1. Read the problem and say your plan aloud before you write any code.
2. Use the AI prompt given (or your own narrow one) to fill in the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference solution only after you have tried.

The ideas behind these problems are in `03_access_control_and_audit_THEORY.md`, next to this notebook.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 12: Filter RAG Chunks by User Permissions

**The situation**

An internal assistant searches HR, finance and engineering documents. Retrieval returns the closest chunks, not the allowed ones. Before any chunk goes into the prompt, you must remove the ones this user may not see, and anything that belongs to another company (tenant).

**Your task**

Write `filter_authorized_chunks(chunks, tenant_id, user_roles)`. Keep a chunk only if it belongs to the user's tenant and either shares a role with the user or is marked public.

**Example**

chunk `{"tenant_id": "t1", "roles": ["finance"]}`, user tenant `"t1"`, user roles `{"engineering"}` -> `[]` (excluded)

**Ask before coding**

- If a chunk has no roles listed, is it private or public?
- Are there deny rules that override allow rules?
- Is the tenant ID always present on chunks?

**Plan to say aloud**

1. Loop over the chunks.
2. Skip any chunk whose tenant_id is not the user's tenant.
3. Keep it if `public` is exactly True.
4. Otherwise keep it only if its roles and the user's roles overlap.
5. Missing or empty roles mean no access. This is failing closed.

### Prompt for your AI assistant

```text
Write a Python function `filter_authorized_chunks(chunks: list[dict], tenant_id: str, user_roles: set[str]) -> list[dict]`. Keep a chunk only if chunk.get("tenant_id") == tenant_id AND (chunk.get("public") is True OR the chunk's "roles" list shares at least one role with user_roles). Missing, None or empty "roles" means not allowed unless public. Keep the original order. Use `.get()` so missing keys never raise. Show only the function.
```

**Check the AI's code for**

- Treats a chunk with no roles as public (fail open). Missing access data must mean no access.
- Checks roles but forgets the tenant check, so another company's chunk can leak.
- Tests `chunk.get("public")` for truthiness, so the string `"false"` counts as public.
- Crashes on `roles: None` because it calls `set(None)`.

In [ ]:
def filter_authorized_chunks(chunks: list[dict], tenant_id: str, user_roles: set[str]) -> list[dict]:
    """Keep only chunks this user may see, in the original order.

    Rules:
    - chunk["tenant_id"] must equal tenant_id (missing means drop).
    - Then keep it if chunk["public"] is True, or its "roles" share a role with user_roles.
    - Missing, None or empty roles mean private (fail closed).
    """
    raise NotImplementedError

In [ ]:
fin = {"id": "c1", "tenant_id": "t1", "roles": ["finance"]}
eng_fin = {"id": "c2", "tenant_id": "t1", "roles": ["engineering", "finance"]}
other_tenant = {"id": "c3", "tenant_id": "t2", "roles": ["finance"]}
no_roles = {"id": "c4", "tenant_id": "t1"}
public = {"id": "c5", "tenant_id": "t1", "public": True}
fake_public = {"id": "c6", "tenant_id": "t1", "public": "false", "roles": None}

cases = [
    (([fin], "t1", {"engineering"}), []),
    (([fin, eng_fin], "t1", {"finance"}), [fin, eng_fin]),
    (([other_tenant], "t1", {"finance"}), []),
    (([no_roles, public], "t1", {"finance"}), [public]),
    (([fake_public], "t1", {"engineering"}), []),
    (([public], "t2", {"finance"}), []),
]
run_checks(filter_authorized_chunks, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def filter_authorized_chunks_reference(chunks: list[dict], tenant_id: str, user_roles: set[str]) -> list[dict]:
    allowed = []
    for chunk in chunks:
        if chunk.get("tenant_id") != tenant_id:
            continue  # never cross tenants
        chunk_roles = set(chunk.get("roles") or [])
        if chunk.get("public") is True or chunk_roles & user_roles:
            allowed.append(chunk)
    return allowed


assert run_checks(filter_authorized_chunks_reference, cases)

**Explain it in 30 seconds**

> "I check the tenant first: a chunk from another company is never allowed, whatever its roles. Then I keep a chunk only if it is clearly public or the user has at least one of its roles. If the access data is missing, I treat the chunk as private. A missing label should cost us an answer, not leak a document, and this runs before the prompt is built because the model cannot un-see text."

**Edge cases to mention**

- A chunk with no tenant_id is dropped.
- Missing or empty roles mean private unless public is True.
- `public` must be the real value True, not a string like "false".
- Access lists can be stale if they are synced only once a day.

**If they ask "how would this work in production?"**

- Filter inside the search itself (per-tenant indexes or metadata filters) so forbidden chunks are never fetched.
- Sync roles from the customer's identity provider and watch how fresh that sync is.
- Log denied chunks with the request ID so security teams can review access attempts.

## Problem 17: Audit Log Generation

**The situation**

A GenAI agent can call tools in a customer's systems, like sending email or updating a ticket. Compliance asks: who did what, when, and in which request? Every tool call needs an audit event, and secrets in the arguments must never end up in the log.

**Your task**

Write `create_audit_event(tenant_id, user_id, tool_name, args, request_id, now_ms=None)`. It returns a dict with those fields, a timestamp, and a copy of args where secret values are replaced with `"[REDACTED]"`, including inside nested dicts and lists.

**Example**

`create_audit_event("t1", "u1", "send_email", {"to": "a@b.com", "api_key": "abc"}, "r1", 1000)` -> `{"tenant_id": "t1", "user_id": "u1", "tool_name": "send_email", "args": {"to": "a@b.com", "api_key": "[REDACTED]"}, "request_id": "r1", "timestamp_ms": 1000}`

**Ask before coding**

- Which fields are required, and what happens if the request ID is missing?
- Which argument names count as secrets?
- Should we also record whether the tool call succeeded?

**Plan to say aloud**

1. Refuse to build the event if tenant, user or request ID is empty (raise ValueError).
2. Walk the args. If a key contains a secret word (password, token, secret, api_key), replace its value.
3. Do the same inside nested dicts and lists.
4. Use the given time, or the current time in milliseconds.
5. Return a new dict and never change the caller's args.

### Prompt for your AI assistant

```text
Write a Python function `create_audit_event(tenant_id: str, user_id: str, tool_name: str, args: dict, request_id: str, now_ms: int | None = None) -> dict`. Raise ValueError if tenant_id, user_id or request_id is empty. Return a dict with keys tenant_id, user_id, tool_name, args, request_id and timestamp_ms. timestamp_ms is now_ms if given, else int(time.time() * 1000). In args, replace the value of any key whose lowercase name contains "password", "token", "secret" or "api_key" with "[REDACTED]". Apply this inside nested dicts and lists too. Build new dicts; do not modify the input args. Show only the function and any small helper it needs.
```

**Check the AI's code for**

- Checks exact key names only, so `db_password` or `API_KEY` slips through.
- Redacts only the top level, leaking a secret inside a nested `auth` dict.
- Changes the caller's args in place, so the real tool call later receives `[REDACTED]`.
- Calls `time.time()` with no way to pass a fixed time, which makes the event hard to test.

In [ ]:
import time

SECRET_WORDS = ("password", "token", "secret", "api_key")


def create_audit_event(tenant_id: str, user_id: str, tool_name: str, args: dict,
                       request_id: str, now_ms: int | None = None) -> dict:
    """Build an audit event for one tool call.

    Rules:
    - Raise ValueError if tenant_id, user_id or request_id is empty.
    - Keys: tenant_id, user_id, tool_name, args, request_id, timestamp_ms.
    - timestamp_ms = now_ms if given, else int(time.time() * 1000).
    - In a COPY of args, any key whose lowercase name contains a SECRET_WORDS entry
      gets the value "[REDACTED]". Also inside nested dicts and lists.
    """
    raise NotImplementedError

In [ ]:
cases = [
    (("t1", "u1", "send_email", {"to": "a@b.com", "api_key": "abc"}, "r1", 1000),
     {"tenant_id": "t1", "user_id": "u1", "tool_name": "send_email",
      "args": {"to": "a@b.com", "api_key": "[REDACTED]"}, "request_id": "r1", "timestamp_ms": 1000}),
    (("t1", "u1", "db_query", {"auth": {"Password": "x"}, "rows": [{"session_token": "s"}], "limit": 5}, "r2", 2000),
     {"tenant_id": "t1", "user_id": "u1", "tool_name": "db_query",
      "args": {"auth": {"Password": "[REDACTED]"}, "rows": [{"session_token": "[REDACTED]"}], "limit": 5},
      "request_id": "r2", "timestamp_ms": 2000}),
    (("t1", "u2", "noop", {}, "r3", 3000),
     {"tenant_id": "t1", "user_id": "u2", "tool_name": "noop", "args": {}, "request_id": "r3", "timestamp_ms": 3000}),
]
if run_checks(create_audit_event, cases):
    try:
        original = {"token": "keep-me"}
        create_audit_event("t1", "u1", "x", original, "r9", 1)
        print("Input args unchanged: ok" if original == {"token": "keep-me"} else "Input args were changed in place.")
        create_audit_event("t1", "u1", "x", {}, "", 1)
        print("Missing request_id was accepted. It should raise ValueError.")
    except ValueError:
        print("Missing request_id rejected: ok")
    except Exception as e:
        print(f"Extra check raised {type(e).__name__}: {e}")

### Reference solution

Try first. Then compare.

In [ ]:
import time

SECRET_WORDS = ("password", "token", "secret", "api_key")


def _redact(value):
    if isinstance(value, dict):
        return {
            key: "[REDACTED]" if any(word in str(key).lower() for word in SECRET_WORDS) else _redact(val)
            for key, val in value.items()
        }
    if isinstance(value, list):
        return [_redact(item) for item in value]
    return value


def create_audit_event_reference(tenant_id: str, user_id: str, tool_name: str, args: dict,
                                 request_id: str, now_ms: int | None = None) -> dict:
    if not tenant_id or not user_id or not request_id:
        raise ValueError("tenant_id, user_id and request_id are required")
    return {
        "tenant_id": tenant_id,
        "user_id": user_id,
        "tool_name": tool_name,
        "args": _redact(args),
        "request_id": request_id,
        "timestamp_ms": now_ms if now_ms is not None else int(time.time() * 1000),
    }


assert run_checks(create_audit_event_reference, cases)
original = {"token": "keep-me"}
create_audit_event_reference("t1", "u1", "x", original, "r9")
assert original == {"token": "keep-me"}, "input args must not change"
try:
    create_audit_event_reference("t1", "u1", "x", {}, "")
    raise AssertionError("missing request_id should raise ValueError")
except ValueError:
    print("Missing request_id rejected: ok")

**Explain it in 30 seconds**

> "I build one flat, predictable event with tenant, user, tool, request ID and a timestamp. Before the args go in, I copy them and replace anything whose key looks like a secret, including inside nested objects. If tenant, user or request ID is missing I refuse, because an audit event you cannot trace back is worthless. The time can be passed in, which keeps the function easy to test."

**Edge cases to mention**

- Secrets under unusual names, like `auth_header` or `pwd`, are missed by a word list.
- Secrets can hide in values, such as a token pasted into a message body.
- A missing request ID raises instead of writing a half-empty event.
- Very large args should be trimmed or summarised before logging.

**If they ask "how would this work in production?"**

- Write events to append-only storage that nobody, including the agent, can edit or delete.
- Add the outcome, latency, approver and policy decision, so the record shows why the call was allowed.
- Use one shared redaction helper across services, so every log is cleaned the same way.

## Wrap-up

- Both treat the tenant as a hard wall: nothing crosses it, and nothing is logged without it.
- Both fail closed: missing access data means no access, and missing trace IDs mean no event.
- Both protect data before it moves on: filter before the prompt, redact before the log.